##### Copyright 2021 Google LLC.

In [1]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a href="https://colab.research.google.com/github/google-research/vision_transformer/blob/master/vit_jax_augreg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## How to train your ViT? Data, Augmentation, and Regularization in Vision Transformers

Model repository published with the paper

[**How to train your ViT? Data, Augmentation, and Regularization in Vision
Transformers**](https://arxiv.org/abs/2106.10270)

This Colab shows how to
[find checkpoints](#scrollTo=F4SLGDtFxlsC)
in the repository, how to
[select and load a model](#scrollTo=wh_SLkQtQ6K4)
form the repository and use it for inference
([also with PyTorch](#scrollTo=1nMyWmDycpAo)),
and how to
[fine-tune on a dataset](#scrollTo=iAruT3YOxqB6).

For more details, please refer to the repository:

https://github.com/google-research/vision_transformer/

Note that this Colab directly uses the unmodified code from the repository. If
you want to modify the modules and persist your changes, you can do all that
using free GPUs and TPUs without leaving the Colab environment - see

https://colab.research.google.com/github/google-research/vision_transformer/blob/master/vit_jax.ipynb

### Imports

In [2]:
# Fetch vision_transformer repository.
![ -d vision_transformer ] || git clone --depth=1 https://github.com/google-research/vision_transformer


Cloning into 'vision_transformer'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 49 (delta 8), reused 35 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 2.02 MiB | 10.11 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [3]:
!pip install tensorflow==2.15.0
!pip install tensorflow-datasets==4.9.3
!pip install protobuf==3.20.3

ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.15.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 40.8 MB/s eta 0:00:00
  Attempting uninstall: tensorflow-datasets
    Found existing installation: tensorflow-datasets 4.9.9
    Uninstalling tensorflow-datasets-4.9.9:
      Successfully uninstalled tensorflow-datasets-4.9.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is th

In [4]:
# Install dependencies.
!pip install -qr vision_transformer/vit_jax/requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 901.6/901.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=

In [5]:
!pip install protobuf==3.20.3
import jax
print(jax.__version__)
import tensorflow as tf
import tensorflow_datasets as tfds

print(tf.__version__)
print(tfds.__version__)



  Using cached protobuf-3.20.3-py2.py3-none-any.whl.metadata (720 bytes)
Using cached protobuf-3.20.3-py2.py3-none-any.whl (162 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.1
    Uninstalling protobuf-7.34.1:
      Successfully uninstalled protobuf-7.34.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-cpu 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 3.20.3 which is incompatible.
google-cloud-secret-manager 2.27.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-translate 3.26.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-pubsub 2.37.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-resource-manager 1.17.0 requires protobuf<8.0.0,>=4.25.8, b

0.7.2


ImportError: cannot import name 'runtime_version' from 'google.protobuf' (/usr/local/lib/python3.12/dist-packages/google/protobuf/__init__.py)

In [ ]:
# Import files from repository.

import sys
if './vision_transformer' not in sys.path:
  sys.path.append('./vision_transformer')

# %load_ext autoreload
# %autoreload 2

from vit_jax import checkpoint
from vit_jax import models
from vit_jax import train
from vit_jax.configs import augreg as augreg_config
from vit_jax.configs import models as models_config

In [ ]:
# Connect to TPUs if runtime type is of type TPU.

import os
if 'google.colab' in str(get_ipython()) and 'COLAB_TPU_ADDR' in os.environ:
  import jax
  import jax.tools.colab_tpu
  jax.tools.colab_tpu.setup_tpu()
  print('Connected to TPU.')
else:
  # Otherwise print information about GPU.
  !nvidia-smi

In [ ]:
# Some more imports used in this Colab.

import glob
import os
import random
import shutil
import time

from absl import logging
import pandas as pd
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from matplotlib import pyplot as plt

pd.options.display.max_colwidth = None
logging.set_verbosity(logging.INFO)  # Shows logs during training.

### Explore checkpoints

This section contains shows how to use the `index.csv` table for model
selection.

See
[`vit_jax.checkpoint.get_augreg_df()`](https://github.com/google-research/vision_transformer/blob/ed1491238f5ff6099cca81087c575a215281ed14/vit_jax/checkpoint.py#L181-L228)
for a detailed description of the individual columns

In [ ]:
# Load master table from Cloud.
with tf.io.gfile.GFile('gs://vit_models/augreg/index.csv') as f:
  df = pd.read_csv(f)

In [ ]:
# This is a pretty large table with lots of columns:
print(f'loaded {len(df):,} rows')
df.columns

In [ ]:
# Number of distinct checkpoints
len(tf.io.gfile.glob('gs://vit_models/augreg/*.npz'))

In [ ]:
# Any column prefixed with "adapt_" pertains to the fine-tuned checkpoints.
# Any column without that prefix pertains to the pre-trained checkpoints.
len(set(df.filename)), len(set(df.adapt_filename))

In [ ]:
df.name.unique()

In [ ]:
# Upstream AugReg parameters (section 3.3):
(
df.groupby(['ds', 'name', 'wd', 'do', 'sd', 'aug']).filename
  .count().unstack().unstack().unstack()
  .dropna(axis=1, how='all').fillna(0).astype(int)
  .iloc[:7]  # Just show beginning of a long table.
)

In [ ]:
# Downstream parameters (table 4)
# (Imbalance in 224 vs. 384 is due to recently added B/8 checkpoints)
(
df.groupby(['adapt_resolution', 'adapt_ds', 'adapt_lr', 'adapt_steps']).filename
  .count().astype(str).unstack().unstack()
  .dropna(axis=1, how='all').fillna('')
)

In [ ]:
# Let's first select the "best checkpoint" for every model. We show in the
# paper (section 4.5) that one can get a good performance by simply choosing the
# best model by final pre-train validation accuracy ("final-val" column).
# Pre-training with imagenet21k 300 epochs (ds=="i21k") gives the best
# performance in almost all cases (figure 6, table 5).
best_filenames = set(
    df.query('ds=="i21k"')
    .groupby('name')
    .apply(lambda df: df.sort_values('final_val').iloc[-1])
    .filename
)

# Select all finetunes from these models.
best_df = df.loc[df.filename.apply(lambda filename: filename in best_filenames)]

# Note: 9 * 68 == 612
len(best_filenames), len(best_df)

In [ ]:
best_df.columns

In [ ]:
# Note that this dataframe contains the models from the "i21k_300" column of
# table 3:
best_df.query('adapt_ds=="imagenet2012"').groupby('name').apply(
    lambda df: df.sort_values('adapt_final_val').iloc[-1]
)[[
   # Columns from upstream
   'name', 'ds', 'filename',
   # Columns from downstream
   'adapt_resolution', 'infer_samples_per_sec','adapt_ds', 'adapt_final_test', 'adapt_filename',
]].sort_values('infer_samples_per_sec')

In [ ]:
# Visualize the 2 (resolution) * 9 (models) * 8 (lr, steps) finetunings for a
# single dataset (Pets37).
# Note how larger models get better scores up to B/16 @384 even on this tiny
# dataset, if pre-trained sufficiently.
sns.relplot(
    data=best_df.query('adapt_ds=="oxford_iiit_pet"'),
    x='infer_samples_per_sec',
    y='adapt_final_val',
    hue='name',
    style='adapt_resolution'
)
plt.gca().set_xscale('log');

In [ ]:
# More details for a single pre-trained checkpoint.
best_df.query('name=="R26+S/32" and adapt_ds=="oxford_iiit_pet"')[[
  col for col in best_df.columns if col.startswith('adapt_')
]].sort_values('adapt_final_val')

### Load a checkpoint

In [ ]:
# Select a value from "adapt_filename" above that is a fine-tuned checkpoint.
# filename = 'R26_S_32-i21k-300ep-lr_0.001-aug_light1-wd_0.1-do_0.0-sd_0.0--oxford_iiit_pet-steps_0k-lr_0.003-res_224'
filename = 'B_16-i1k-300ep-lr_0.001-aug_light0-wd_0.03-do_0.0-sd_0.0--cifar100-steps_10k-lr_0.001-res_224'
tfds_name = filename.split('--')[1].split('-')[0]
# model_config = models_config.AUGREG_CONFIGS[filename.split('-')[0]]
model_config = models_config.AUGREG_CONFIGS['B_16']
# resolution = int(filename.split('_')[-1])
resolution = 224
path = f'gs://vit_models/augreg/{filename}.npz'


print(f'{tf.io.gfile.stat(path).length / 1024 / 1024:.1f} MiB - {path}')

In [ ]:
import tensorflow as tf

files = tf.io.gfile.glob("gs://vit_models/augreg/*.npz")

print(len(files))
print(files[:20])

In [ ]:
import jax
import jax.numpy as jnp
import tensorflow as tf
import tensorflow_datasets as tfds

from vit_jax import models, checkpoint
from vit_jax.configs import models as models_config


ds, ds_info = tfds.load(
    'cifar100',
    split='train',
    as_supervised=True,
    with_info=True
)

num_classes = ds_info.features['label'].num_classes

def preprocess(image, label):
    image = tf.image.resize(image, (resolution, resolution))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

ds = ds.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

# take one batch
images, labels = next(iter(ds))
images = jnp.array(images)



In [ ]:
# 3. MODEL
model_config = models_config.AUGREG_CONFIGS['B_16']

model = models.VisionTransformer(
    num_classes=num_classes,
    **model_config
)

# init params (required for shape matching)
init_params = model.init(
    jax.random.PRNGKey(0),
    jnp.ones([1, resolution, resolution, 3]),
    train=False
)['params']



In [ ]:
# 4. LOAD CHECKPOINT
params = checkpoint.load(path)

# 5. INFERENCE
logits = model.apply(
    {'params': params},
    images,
    train=False
)

# 6. DEBUG CHECKS
print("logits shape:", logits.shape)
print("min:", logits.min())
print("max:", logits.max())
print("std:", logits.std())

# 7. PREDICTION
probs = jax.nn.softmax(logits)
preds = jnp.argmax(probs, axis=-1)

print("preds:", preds[:10])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# pick one sample from batch
idx = 0
logit = np.array(logits[idx])   # (100,)
prob = np.array(probs[idx])     # (100,)

class_names = [ds_info.features['label'].int2str(i) for i in range(num_classes)]

# -----------------------
# Plot logits
# -----------------------
plt.figure(figsize=(12, 4))
plt.bar(range(num_classes), logit)
plt.title(f"Logits (sample {idx})")
plt.xlabel("Class index")
plt.ylabel("Logit value")
plt.show()



In [ ]:
# -----------------------
# Plot probabilities (better interpretation)
# -----------------------
plt.figure(figsize=(12, 4))
plt.bar(range(num_classes), prob)
plt.title(f"Probabilities (sample {idx})")
plt.xlabel("Class index")
plt.ylabel("Probability")
plt.show()

# -----------------------
# Top-10 predictions (clean view)
# -----------------------
topk = 10
top_idx = np.argsort(prob)[-topk:][::-1]

print("\nTop predictions:")
for i in top_idx:
    print(class_names[i], ":", prob[i])

In [ ]:
import matplotlib.pyplot as plt

label_names = ds_info.features['label'].names

def show_results(images, labels, preds, n=8):
    plt.figure(figsize=(12, 6))

    for i in range(n):
        plt.subplot(2, 4, i + 1)

        plt.imshow(images[i])

        true_label = label_names[int(labels[i])]
        pred_label = label_names[int(preds[i])]

        color = "green" if labels[i] == preds[i] else "red"

        plt.title(f"T: {true_label}\nP: {pred_label}", color=color)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# convert labels to jax (important fix)
labels = jnp.array(labels)

# show results
show_results(images, labels, preds)

In [ ]:
def show_images(images, labels, preds, n=8):
    plt.figure(figsize=(12, 6))

    for i in range(n):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i])

        true_label = label_names[int(labels[i])]
        pred_label = label_names[int(preds[i])]

        color = "green" if labels[i] == preds[i] else "red"

        plt.title(f"T:{true_label}\nP:{pred_label}", color=color)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_images(images_list, labels_list, preds_list)

In [ ]:
# Fetch dataset that the checkpoint was finetuned on.
# (Note that automatic download does not work with imagenet2012)
# ds = tfds.load('cifar100', split='train', with_info=True)
ds, ds_info = tfds.load('cifar100', split='train', with_info=True)

In [ ]:
# Get model instance - no weights are initialized yet.
# model = models.VisionTransformer(
#     num_classes=ds_info.features['label'].num_classes, **model_config)

model_config = models_config.AUGREG_CONFIGS['B_16']
model = models.VisionTransformer(
    num_classes=100,
    **model_config
)

In [ ]:
# Load a checkpoint from cloud - for large checkpoints this can take a while...
# params = checkpoint.load(path)

import jax
import jax.numpy as jnp
from vit_jax import checkpoint

init_params = model.init(
    jax.random.PRNGKey(0),
    jnp.ones([1, resolution, resolution, 3]),
    train=False
)['params']
params = checkpoint.load(path)
params = checkpoint.load_pretrained(
    pretrained_path=path,
    init_params=init_params,
    model_config=model_config
)


In [ ]:
# Added by me
def preprocess(example):
    image = example['image']
    image = tf.image.resize(image, (224, 224))
    image = tf.cast(image, tf.float32) / 255.0
    return image, example['label']

In [ ]:
import jax.numpy as jnp

batch = next(iter(ds))
images = jnp.array(batch[0])
labels = jnp.array(batch[1])

In [ ]:
import optax
import flax.linen as nn

optimizer = optax.adam(1e-4)
opt_state = optimizer.init(params)

@jax.jit
def train_step(params, opt_state, images, labels):

    def loss_fn(params):
        logits = model.apply({'params': params}, images, train=True)
        loss = optax.softmax_cross_entropy_with_integer_labels(logits, labels).mean()
        return loss

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    return params, opt_state, loss

In [ ]:
def pp(img, sz):
  """Simple image preprocessing."""
  img = tf.cast(img, float) / 255.0
  img = tf.image.resize(img, [sz, sz])
  return img

plt.imshow(pp(d['image'], resolution));

In [ ]:
# Get a single example from dataset for inference.
d = next(iter(ds['test']))

In [ ]:
# Inference on batch with single example.
# logits, = model.apply({'params': params}, pp(d['image'], resolution).numpy()[None], train=False)
# logits = model.apply({'params': params}, image[None], train=False)
logits = model.apply(
    {'params': params},
    images,
    train=False
)
print(logits.shape)

In [ ]:
print("shape:", images.shape)
print("dtype:", images.dtype)
print("min:", images.min())
print("max:", images.max())
print("mean:", images.mean())

In [ ]:
# Added by me
import jax.nn as jnn
probs = jnn.softmax(logits)
print(probs)
print(type(logits))
print(logits.shape)
print(logits)

In [ ]:
pred = jnp.argmax(logits, axis=-1)
acc = jnp.mean(pred == labels)
print(acc)

In [ ]:
# Plot logits (you can use tf.nn.softmax() to show probabilities instead).
plt.figure(figsize=(10, 4))
plt.bar(list(map(ds_info.features['label'].int2str, range(len(logits)))), logits)
plt.xticks(rotation=90);

#### Using `timm`

If you know PyTorch, you're probably already familiar with `timm`.

If not yet - it's your lucky day! Please check out their docs here:

https://rwightman.github.io/pytorch-image-models/

In [ ]:
# Checkpoints can also be loaded directly into timm...
!pip install timm
import timm
import torch

In [ ]:
# For available model names, see here:
# https://github.com/rwightman/pytorch-image-models/blob/master/timm/models/vision_transformer.py
# https://github.com/rwightman/pytorch-image-models/blob/master/timm/models/vision_transformer_hybrid.py
timm_model = timm.create_model(
    'vit_small_r26_s32_384', num_classes=ds_info.features['label'].num_classes)

# Non-default checkpoints need to be loaded from local files.
if not tf.io.gfile.exists(f'{filename}.npz'):
  tf.io.gfile.copy(f'gs://vit_models/augreg/{filename}.npz', f'{filename}.npz')
timm.models.load_checkpoint(timm_model, f'{filename}.npz')

In [ ]:
def pp_torch(img, sz):
  """Simple image preprocessing for PyTorch."""
  img = pp(img, sz)
  img = img.numpy().transpose([2, 0, 1])  # PyTorch expects NCHW format.
  return torch.tensor(img[None])

with torch.no_grad():
  logits, = timm_model(pp_torch(d['image'], resolution)).detach().numpy()

In [ ]:
# Same results as above (since we loaded the same checkpoint).
plt.figure(figsize=(10, 4))
plt.bar(list(map(ds_info.features['label'].int2str, range(len(logits)))), logits)
plt.xticks(rotation=90);

### Fine-tune

You want to be connected to a TPU or GPU runtime for fine-tuning.

Note that here we're just calling into the code. For more details see the
annotated Colab

https://colab.research.google.com/github/google-research/vision_transformer/blob/linen/vit_jax.ipynb

Also note that Colab GPUs and TPUs are not very powerful. To run this code on
more powerful machines, see:

https://github.com/google-research/vision_transformer/#running-on-cloud

In particular, note that due to the Colab "TPU Node" setup, transfering data to
the TPUs is realtively slow (for example the smallest `R+Ti/16` model trains
faster on a single GPU than on 8 TPUs...)

#### TensorBoard

In [ ]:
# Launch tensorboard before training - maybe click "reload" during training.
%load_ext tensorboard
%tensorboard --logdir=./workdirs

#### From tfds

In [ ]:
# Create a new temporary workdir.
workdir = f'./workdirs/{int(time.time())}'
workdir

In [ ]:
# Get config for specified model.

# Note that we can specify simply the model name (in which case the recommended
# checkpoint for that model is taken), or it can be specified by its full
# name.
config = augreg_config.get_config('B_16')

# A very small tfds dataset that only has a "train" split. We use this single
# split both for training & evaluation by splitting it further into 90%/10%.
config.dataset = 'tf_flowers'
config.pp.train = 'train[:90%]'
config.pp.test = 'train[90%:]'
# tf_flowers only has 3670 images - so the 10% evaluation split will contain
# 360 images. We specify batch_eval=120 so we evaluate on all but 7 of those
# images (remainder is dropped).
config.batch_eval = 120

# Some more parameters that you will often want to set manually.
# For example for VTAB we used steps={500, 2500} and lr={.001, .003, .01, .03}
config.base_lr = 0.01
config.shuffle_buffer = 1000
config.total_steps = 100
config.warmup_steps = 10
config.accum_steps = 0  # Not needed with R+Ti/16 model.
config.pp['crop'] = 224

In [ ]:
# Call main training loop. See repository and above Colab for details.
state = train.train_and_evaluate(config, workdir)

#### From JPG files

The codebase supports training directly form JPG files on the local filesystem
instead of `tfds` datasets. Note that the throughput is somewhat reduced, but
that only is noticeable for very small models.

The main advantage of `tfds` datasets is that they are versioned and available
globally.

In [ ]:
base = '.'  # Store data on VM (ephemeral).

In [ ]:
# Uncomment below lines if you want to download & persist files in your Google
# Drive instead. Note that Colab VMs are reset (i.e. files are deleted) after
# some time of inactivity. Storing data to Google Drive guarantees that it is
# still available next time you connect from a new VM.

# Note that this is significantly slower than reading from the VMs locally
# attached file system!

# from google.colab import drive
# drive.mount('/gdrive')
# base = '/gdrive/My Drive/vision_transformer_images'

In [ ]:
# Download some dataset & unzip.
! rm -rf $base/flower_photos; mkdir -p $base
! (cd $base && curl https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz | tar xz)

In [ ]:
# Since the default file format of above "tf_flowers" dataset is
# flower_photos/{class_name}/{filename}.jpg
# we first need to split it into a "train" (90%) and a "test" (10%) set:
# flower_photos/train/{class_name}/{filename}.jpg
# flower_photos/test/{class_name}/{filename}.jpg

def split(base_dir, test_ratio=0.1):
  paths = glob.glob(f'{base_dir}/*/*.jpg')
  random.shuffle(paths)
  counts = dict(test=0, train=0)
  for i, path in enumerate(paths):
    split = 'test' if i < test_ratio * len(paths) else 'train'
    *_, class_name, basename = path.split('/')
    dst = f'{base_dir}/{split}/{class_name}/{basename}'
    if not os.path.isdir(os.path.dirname(dst)):
      os.makedirs(os.path.dirname(dst))
    shutil.move(path, dst)
    counts[split] += 1
  print(f'Moved {counts["train"]:,} train and {counts["test"]:,} test images.')

split(f'{base}/flower_photos')

In [ ]:
# Create a new temporary workdir.
workdir = f'./workdirs/{int(time.time())}'
workdir

In [ ]:
# Read data from directory containing files.
# (See cell above for more config settings)
config.dataset = f'{base}/flower_photos'

In [ ]:
# And fine-tune on images provided
opt = train.train_and_evaluate(config, workdir)